# 01. Heaps

**Tier:** Deep. **Why you'd use this:** You repeatedly need the two shortest sticks, glue them, pay their sum, and put the joined stick back. A min-heap keeps the shortest stick at the front after every change.

**After this lesson you can:** heapify a list; pop the two smallest; push a combined value back until one remains.
About 25 minutes. **Uses:** [heapq](../../course/08-collections-module/04-heapq-lesson.ipynb), [Lists](../../course/04-collections/01-lists-lesson.ipynb)


## Quick review

1. Where does the smallest value sit after `heapq.heapify`?
2. What do `heappop` and `heappush` each cost on a heap of size `n`?

<details><summary>Answers</summary>

Index `0`. About `log n` time each. Skim [heapq](../../course/08-collections-module/04-heapq-lesson.ipynb) if the API is new.
</details>


## Recognize it

You repeatedly need the current minimum (or two minima), update the collection, and ask again. The prompt talks about "always take the smallest", "priority queue", or "join cheapest first".

Signals: many extract-min operations interleaved with inserts. A look-alike that is not this pattern: sorting once and reading in order when nothing is ever inserted back.


## Why it works this way

Each join needs the two shortest sticks among those that remain. Sorting the whole list before every join would resort about `n` times and pay roughly `n² log n` work. A binary heap keeps the minimum at the root and restores that property after each pop or push in logarithmic time. `heapify` builds that shape from an unsorted list in linear time.

The algorithm: heapify the stick lengths; while more than one stick remains, pop twice, add the sum to the total cost, and push the sum back as a new stick. The cost definition matches the problem: you pay the length of the stick you create at each join. Stopping with one stick left means nothing remains to join.

Why greedily joining the two shortest is optimal is a separate exchange argument: any plan that delays a small join can be rearranged without increasing the total. The heap is the data structure that makes the greedy choice cheap to implement. Extra memory is the heap of remaining sticks, at most `n`. Total time is about `n log n` because each stick is pushed and popped a constant number of times across the process.

`heapq` is min-only. For "two heaviest" problems you negate numbers on the way in and out. `nsmallest(k)` is the right helper when you only need k values once and will not push results back.
Last-stone smashing is the max-heap cousin: always smash the two heaviest, push the difference if nonzero. Negate weights into a min-heap to get max-heap behavior with `heapq`. K-smallest can also keep a max-heap of size k while you scan: push until full, then replace the max when a smaller value arrives. That uses `O(k)` memory instead of `O(n)` when k is tiny.

Tie-breaking with tuples `(priority, tie, item)` keeps comparisons stable when priorities collide. Never compare incomparable payloads on the heap without a tie field. For teaching join cost, plain integers are enough and keep the trace readable.


## When to use it

Use a heap when you interleave extract-min with inserts. Prefer a full sort when the multiset never changes after the first ordering. Prefer `nsmallest` / `nlargest` for a one-shot top-k. Prefer a balanced tree only when you also need arbitrary deletes by identity.
 Streaming inputs that never fit in memory need a bounded heap of size k and a single pass; loading every stick first assumes the list fits.


## Brute force, and why it's slow

Each step sort the remaining sticks, take the two shortest, and rebuild the list.


In [1]:
def join_cost_slow(sticks):
    sorts = 0
    sticks = list(sticks)
    cost = 0
    while len(sticks) > 1:
        sticks.sort()
        sorts += 1
        first, second, *rest = sticks
        joined = first + second
        cost += joined
        sticks = rest + [joined]
    return cost, sorts


for size in (20, 40, 60):
    print(size, join_cost_slow(list(range(1, size + 1))))

20 (864, 19)
40 (4180, 39)
60 (10368, 59)


## Key insight

A min-heap always exposes the shortest stick in constant time and restores order in logarithmic time after each pop or push. Join the two shortest until one stick remains.


## Worked example

Repeatedly join the two shortest sticks. Each join costs the sum of the two lengths. Return the total cost.


In [2]:
import heapq


def join_cost(sticks: list[int]) -> int:
    # 1. Min-heap of remaining stick lengths.
    heap = list(sticks)
    heapq.heapify(heap)
    cost = 0
    # 2. Join until one stick is left.
    while len(heap) > 1:
        first = heapq.heappop(heap)
        second = heapq.heappop(heap)
        joined = first + second
        cost += joined
        heapq.heappush(heap, joined)
    return cost


print(join_cost([1, 8, 3, 5]))
print(join_cost([1, 2]))
print(join_cost([5]))

30
3
0


## Trace

`[1, 8, 3, 5]` after heapify (min at front):

| pop | pop | join | cost | heap grows by |
|---|---|---|---|---|
| 1 | 3 | 4 | 4 | push 4 |
| 4 | 5 | 9 | 13 | push 9 |
| 8 | 9 | 17 | **30** | push 17 |

One stick left; return **30**.


## Cost

About `n` heap operations of `log n` each: time grows like `n log n`. Extra memory is the heap. Resorting every round pays a sort per join.

With `n` sticks you perform roughly `n - 1` joins. Each join does two pops and one push, each `O(log n)`, so the `n log n` bound is tight up to a constant. Sorting every round repeats work the heap already organized after the previous join.


## Variants

Not solved here: [Top K Frequent Elements](../../challenges/problems/15-top-k-frequent-elements.ipynb), [Kth Largest Element in an Array](../../challenges/problems/26-kth-largest-element.ipynb), [Merge k Sorted Lists](../../challenges/problems/40-merge-k-sorted-lists.ipynb).


## Common mistake

This bug is silent. The code pays for each pair it pops but never pushes the joined stick back, so later rounds ignore that stick and undercount. There is no traceback. The print is `17`, and the answer is `30`.


In [3]:
def join_cost_wrong(sticks):
    heap = list(sticks)
    heapq.heapify(heap)
    cost = 0
    while len(heap) > 1:
        first = heapq.heappop(heap)
        second = heapq.heappop(heap)
        cost += first + second
        # forgot: heapq.heappush(heap, first + second)
    return cost


print(join_cost_wrong([1, 8, 3, 5]))

17


The fix pushes `first + second` back onto the heap before the next round. Run `join_cost([1, 8, 3, 5])` and you get `30`.


In [4]:
print(join_cost([1, 8, 3, 5]))

30


## Python idioms

`heapq.heapify` rearranges in place in linear time — cheaper than pushing `n` times. For a one-shot k smallest without reinserts, `heapq.nsmallest(k, sticks)` is enough.


In [5]:
print(heapq.nsmallest(2, [1, 8, 3, 5]))

[1, 3]


## Recap

- A min-heap keeps the shortest stick at the front.
- Pop two, pay the sum, push the sum back.
- Stop when one stick remains; the paid total is the answer.

Next: [practice](01-last-stone-practice.ipynb).
